In [ ]:
pip install -q gymnasium gym-anytrading stable-baselines3


In [ ]:
# Imports
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import gymnasium as gym
import gym_anytrading
from gym_anytrading.envs import Actions
from stable_baselines3 import PPO

# Plot style
plt.style.use('seaborn-v0_8')



In [ ]:
# Load data
csv_path = "/Users/sachabrouck/StockTradingSimDatacamp/AAPL copy.csv"
data = pd.read_csv(csv_path, parse_dates=["Date"])  # keep Date as column first
# Ensure correct columns
assert {"Date", "Close"}.issubset(data.columns), "CSV must contain Date and Close columns"
# Set index to Date
data.set_index("Date", inplace=True)

# Basic preview
display(data.head())
print("Rows:", len(data))



In [ ]:
# Create environment
window_size = 10
start_index = window_size
end_index = len(data)

env = gym.make(
    'stocks-v0',
    df=data,
    window_size=window_size,
    frame_bound=(start_index, end_index)
)
print("Observation Space:", env.observation_space)



In [ ]:
# Train PPO model
model = PPO('MlpPolicy', env, verbose=0)
model.learn(total_timesteps=10_000)
print("Training complete.")



In [ ]:
# Evaluation with percentage-based position sizing
# We'll use the trained policy's actions as signals, but track our own cash and shares.

eval_env = gym.make(
    'stocks-v0',
    df=data,
    window_size=window_size,
    frame_bound=(start_index, end_index)
)
obs, info = eval_env.reset(seed=2024)

initial_cash = 100_000.0
cash = initial_cash
shares_held = 0
balance_history = [cash]

# Trade 10% of current cash (or shares) per step when buying/selling
trade_fraction = 0.10

buy_points = []  # (date, price)
sell_points = [] # (date, price)

# Helper to get current date and price from env internals
prices = eval_env.unwrapped.prices  # numpy array aligned with env's windowed slice
# Dates aligned to the env's prices slice (includes the initial lookback window)
dates_for_prices = data.index[start_index - window_size:end_index]
# Dates used to tag actions (no lookback padding)
index_dates = data.index[start_index:end_index]

terminated = False
truncated = False

while not (terminated or truncated):
    action, _ = model.predict(obs, deterministic=True)

    # Get current tick and price
    current_tick = eval_env.unwrapped._current_tick  # internal but available
    # Map env tick into price array position (prices include lookback window)
    price_idx = current_tick - (start_index - window_size)
    price = float(prices[price_idx])
    date = index_dates[current_tick - start_index]

    # Apply our own trade sizing logic
    if action == Actions.Buy:
        # Spend a fraction of available cash to buy
        budget = cash * trade_fraction
        num_shares = int(budget // price)
        if num_shares > 0:
            cash -= num_shares * price
            shares_held += num_shares
            buy_points.append((date, price))
    elif action == Actions.Sell:
        # Sell a fraction of currently held shares
        shares_to_sell = int(max(1, shares_held * trade_fraction)) if shares_held > 0 else 0
        if shares_to_sell > 0:
            cash += shares_to_sell * price
            shares_held -= shares_to_sell
            sell_points.append((date, price))

    # Mark-to-market portfolio value tracking
    portfolio_value = cash + shares_held * price
    balance_history.append(portfolio_value)

    # Step env forward to get next observation
    obs, reward, terminated, truncated, info = eval_env.step(int(action))

# Liquidate remaining shares at the final price
final_price_idx = min(len(prices) - 1, eval_env.unwrapped._current_tick - (start_index - window_size))
final_price = float(prices[final_price_idx])
if shares_held > 0:
    cash += shares_held * final_price
    sell_points.append((index_dates[-1], final_price))
    shares_held = 0

final_balance = cash
print(f"Final balance: {final_balance:,.2f} (Initial: {initial_cash:,.2f})")



In [ ]:
# Create your two charts below. Note, do not change the fig and ax variable names.

# Chart 1, a plot showing trading actions
fig, ax = plt.subplots()
ax.plot(dates_for_prices, prices, label='Close Price', color='black', linewidth=1.2)
if buy_points:
    b_dates, b_prices = zip(*buy_points)
    ax.scatter(b_dates, b_prices, marker='^', color='green', s=60, label='Buy')
if sell_points:
    s_dates, s_prices = zip(*sell_points)
    ax.scatter(s_dates, s_prices, marker='v', color='red', s=60, label='Sell')
ax.set_title('AAPL Price with Buy/Sell Signals (PPO)')
ax.set_ylabel('Price')
ax.legend(loc='best')
# Save chart 1
import os
os.makedirs('charts', exist_ok=True)
fig.savefig('charts/price_actions.png', dpi=150, bbox_inches='tight')
plt.show()

# Chart 2, a plot of the balance_history over time
fig2, ax2 = plt.subplots()
ax2.plot(range(len(balance_history)), balance_history, color='blue')
ax2.set_title('Portfolio Value Over Time')
ax2.set_xlabel('Step')
ax2.set_ylabel('Portfolio Value ($)')
# Save chart 2
fig2.savefig('charts/portfolio_value.png', dpi=150, bbox_inches='tight')
plt.show()



### Discussion

- **Performance**: With only 10k timesteps and a simple observation space, performance is typically modest and may track trend-following behavior without robust risk controls. Expect high variance and possible overfitting to the period.
- **Why it performed this way**: The `stocks-v0` environment abstracts trading heavily and does not expose rich features (e.g., volume, volatility, technical indicators). PPO needs sufficient signal and training time; here, signals are sparse, and the reward structure is simplistic.
- **Data selection impact**:
  - If the chosen period trends upward, naive buy-and-hold often outperforms; in regimes with mean-reversion or sideways markets, frequent trades can hurt.
  - Lack of multiple regimes and assets limits generalization. Using only `Close` omits critical context (volume, spreads, macro events).
  - Data frequency (daily) reduces opportunities versus intraday but lowers noise; survivorship/selection bias can inflate results.

